# 02 CIFAR-10 Baseline Training (MSE only)

DDPM baseline. Uses only MSE loss without SDD.

- `num_processes=None` → auto-detect GPUs
- Works on a single GPU / CPU environment as well

In [ ]:
import torch
from src.experiments import load_cfg, deep_update

# ── GPU Auto-Detection ───────────────────────────────────────
n_gpu = torch.cuda.device_count()
print(f"Detected number of GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory // 1024**3} GB")
if n_gpu == 0:
    print("  (No GPU — running on CPU)")


In [ ]:
from src.experiments import load_cfg, deep_update, launch_train

cfg = load_cfg("configs/cifar10.yaml")
cfg = deep_update(cfg, {
    "run_name": "cifar10_baseline_mse_only",
    "sdd": {
        "enabled": False,
        "lambda_distill": 0.0,
        "use_centering": False,
        "use_sharpening": False,
        "use_gating": False,
        "use_projection_head": False,
    },
    "wandb": {"enabled": True, "tags": ["cifar10", "baseline", "mse-only"]},
})

# num_processes=None → use all GPUs automatically
launch_train(cfg, num_processes=None, with_eval=True)


In [ ]:
# Check loss curve after training
from pathlib import Path
import pandas as pd, matplotlib.pyplot as plt

csv = Path("outputs/logs/cifar10_baseline_mse_only_history.csv")
if csv.exists():
    df = pd.read_csv(csv)
    ax = df.plot(x="epoch", y=["loss_total", "loss_mse"], figsize=(10, 4),
                 title="Baseline training curves")
    ax.grid(True)
    plt.show()
    print(df.tail())
else:
    print("No training results yet — run the cell above first.")
